# Benchmark: Registration Methods & Real Data

Tests DVFopt as a post-processing step on displacement fields produced by
different registration algorithms:

| Method | Library | Notes |
|--------|---------|-------|
| **Demons** (diffeomorphic) | SimpleITK | Classic iterative, often produces folding |
| **SyN** (symmetric normalization) | ANTsPy | State-of-the-art diffeomorphic |
| **B-spline** (free-form deformation) | SimpleITK | Parametric, can fold at large deformations |
| **VoxelMorph** | voxelmorph + PyTorch | Deep learning, fast inference |

Each method registers the same pair of 2D images.  We then check how many
negative Jacobian determinants each produces and correct with DVFopt.

**Requirements:** Install optional deps as needed:
```
pip install SimpleITK antspyx
```
VoxelMorph requires PyTorch — see `setup-venv.bat`.

In [ ]:
import time
import importlib

import numpy as np
import matplotlib.pyplot as plt

from dvfopt import jacobian_det2D, iterative_parallel
from dvfopt.viz import (
    plot_grid_before_after,
    plot_checkerboard_before_after,
    plot_neg_jdet_neighborhoods,
)

# Check which backends are available
HAS_SITK = importlib.util.find_spec("SimpleITK") is not None
HAS_ANTS = importlib.util.find_spec("ants") is not None
HAS_VXM = importlib.util.find_spec("voxelmorph") is not None

print(f"SimpleITK : {'available' if HAS_SITK else 'MISSING'}")
print(f"ANTsPy    : {'available' if HAS_ANTS else 'MISSING'}")
print(f"VoxelMorph: {'available' if HAS_VXM else 'MISSING'}")

## Configuration

In [ ]:
IMAGE_SIZE = 128  # HxW for synthetic test images
JDET_THRESHOLD = 0.01

# Set to a .nii.gz path pair to use real images instead of synthetic
FIXED_IMAGE_PATH = None   # e.g. "data/fixed.nii.gz"
MOVING_IMAGE_PATH = None  # e.g. "data/moving.nii.gz"

## Generate (or load) test images

Synthetic images use overlapping circles/ellipses with enough deformation
to challenge the registration methods.

In [ ]:
def make_test_image(size, shapes, rng):
    """Create a 2D image with ellipses at specified locations."""
    img = np.zeros((size, size), dtype=np.float64)
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float64)
    for cy, cx, ry, rx, intensity in shapes:
        mask = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1.0
        img[mask] = intensity
    return img


if FIXED_IMAGE_PATH is not None and MOVING_IMAGE_PATH is not None:
    import SimpleITK as sitk
    fixed_img = sitk.ReadImage(FIXED_IMAGE_PATH, sitk.sitkFloat64)
    moving_img = sitk.ReadImage(MOVING_IMAGE_PATH, sitk.sitkFloat64)
    fixed_np = sitk.GetArrayFromImage(fixed_img).astype(np.float64)
    moving_np = sitk.GetArrayFromImage(moving_img).astype(np.float64)
    IMAGE_SIZE = fixed_np.shape[0]
    print(f"Loaded real images: {fixed_np.shape}")
else:
    rng = np.random.default_rng(123)
    S = IMAGE_SIZE

    # Fixed image: centered shapes
    fixed_np = make_test_image(S, [
        (S*0.35, S*0.35, S*0.15, S*0.12, 0.9),
        (S*0.60, S*0.55, S*0.18, S*0.14, 0.7),
        (S*0.30, S*0.65, S*0.10, S*0.10, 0.5),
        (S*0.70, S*0.30, S*0.12, S*0.08, 0.6),
    ], rng)

    # Moving image: shapes shifted/scaled to require non-trivial registration
    moving_np = make_test_image(S, [
        (S*0.30, S*0.40, S*0.18, S*0.10, 0.9),
        (S*0.65, S*0.50, S*0.15, S*0.17, 0.7),
        (S*0.25, S*0.60, S*0.12, S*0.08, 0.5),
        (S*0.72, S*0.25, S*0.10, S*0.12, 0.6),
    ], rng)

    print(f"Generated synthetic {S}x{S} test images")

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(fixed_np, cmap="gray"); axes[0].set_title("Fixed")
axes[1].imshow(moving_np, cmap="gray"); axes[1].set_title("Moving")
for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## Helper: convert displacement field to DVFopt convention

In [ ]:
def to_dvfopt(dy, dx):
    """Convert (dy, dx) displacement arrays to (3, 1, H, W) deformation."""
    H, W = dy.shape
    deformation = np.zeros((3, 1, H, W), dtype=np.float64)
    deformation[1, 0] = dy
    deformation[2, 0] = dx
    return deformation


def summarize_dvf(deformation, label):
    """Print Jacobian stats for a deformation field."""
    phi = np.stack([deformation[1, 0], deformation[2, 0]])
    jac = jacobian_det2D(phi)
    n_neg = int((jac <= 0).sum())
    n_below = int((jac <= JDET_THRESHOLD).sum())
    H, W = deformation.shape[-2:]
    print(f"  {label:<25s}  {H}x{W}  neg={n_neg:>5d}  "
          f"below_thr={n_below:>5d}  min_jdet={jac.min():+.6f}  "
          f"disp_range=[{phi.min():.2f}, {phi.max():.2f}]")
    return n_neg

---
## Method 1: Demons (SimpleITK)

Diffeomorphic demons with multi-resolution.

In [ ]:
deformations = {}  # method_name → (deformation, reg_time)

if HAS_SITK:
    import SimpleITK as sitk

    fixed_sitk = sitk.GetImageFromArray(fixed_np)
    moving_sitk = sitk.GetImageFromArray(moving_np)
    fixed_sitk = sitk.Cast(fixed_sitk, sitk.sitkFloat32)
    moving_sitk = sitk.Cast(moving_sitk, sitk.sitkFloat32)

    # --- Diffeomorphic Demons ---
    demons = sitk.DiffeomorphicDemonsRegistrationFilter()
    demons.SetNumberOfIterations(200)
    demons.SetStandardDeviations(1.5)

    t0 = time.perf_counter()
    disp_field = demons.Execute(fixed_sitk, moving_sitk)
    reg_time = time.perf_counter() - t0

    disp_np = sitk.GetArrayFromImage(disp_field)  # (H, W, 2) for 2D
    # SimpleITK 2D displacement: [..., 0]=dx, [..., 1]=dy
    dy = disp_np[..., 1].astype(np.float64)
    dx = disp_np[..., 0].astype(np.float64)
    deformations["Demons"] = (to_dvfopt(dy, dx), reg_time)
    summarize_dvf(deformations["Demons"][0], "Demons")
    print(f"  Registration time: {reg_time:.2f}s")

    # --- B-spline (Free-Form Deformation) ---
    bspline_reg = sitk.ImageRegistrationMethod()
    bspline_reg.SetMetricAsMattesMutualInformation(numberOfHistogramBins=50)
    bspline_reg.SetOptimizerAsLBFGSB(gradientConvergenceTolerance=1e-5,
                                      numberOfIterations=100)
    grid_spacing = [IMAGE_SIZE // 6] * 2
    tx = sitk.BSplineTransformInitializer(fixed_sitk, grid_spacing, order=3)
    bspline_reg.SetInitialTransform(tx, inPlace=True)
    bspline_reg.SetInterpolator(sitk.sitkLinear)
    bspline_reg.SetShrinkFactorsPerLevel([4, 2, 1])
    bspline_reg.SetSmoothingSigmasPerLevel([2, 1, 0])
    bspline_reg.SmoothingSigmasAreSpecifiedInPhysicalUnitsOn()

    t0 = time.perf_counter()
    final_tx = bspline_reg.Execute(fixed_sitk, moving_sitk)
    reg_time = time.perf_counter() - t0

    # Convert transform to displacement field
    disp_filter = sitk.TransformToDisplacementFieldFilter()
    disp_filter.SetReferenceImage(fixed_sitk)
    disp_sitk = disp_filter.Execute(final_tx)
    disp_np = sitk.GetArrayFromImage(disp_sitk)
    dy = disp_np[..., 1].astype(np.float64)
    dx = disp_np[..., 0].astype(np.float64)
    deformations["B-spline"] = (to_dvfopt(dy, dx), reg_time)
    summarize_dvf(deformations["B-spline"][0], "B-spline")
    print(f"  Registration time: {reg_time:.2f}s")
else:
    print("SimpleITK not available — skipping Demons and B-spline")

## Method 2: SyN (ANTsPy)

In [ ]:
if HAS_ANTS:
    import ants

    fixed_ants = ants.from_numpy(fixed_np.astype(np.float32))
    moving_ants = ants.from_numpy(moving_np.astype(np.float32))

    t0 = time.perf_counter()
    result = ants.registration(
        fixed=fixed_ants,
        moving=moving_ants,
        type_of_transform="SyN",
        syn_metric="mattes",
        syn_sampling=32,
        reg_iterations=(100, 70, 50),
        verbose=False,
    )
    reg_time = time.perf_counter() - t0

    # Load the forward warp field
    fwd_paths = [p for p in result["fwdtransforms"] if p.endswith(".nii.gz")]
    if fwd_paths:
        import nibabel as nib
        warp_nib = nib.load(fwd_paths[0])
        warp_data = warp_nib.get_fdata()
        # ANTs 2D warp: (H, W, 1, 1, 2) → squeeze → (H, W, 2)
        warp_data = warp_data.squeeze()
        if warp_data.ndim == 3 and warp_data.shape[-1] == 2:
            dx = warp_data[..., 0].astype(np.float64)
            dy = warp_data[..., 1].astype(np.float64)
            deformations["SyN"] = (to_dvfopt(dy, dx), reg_time)
            summarize_dvf(deformations["SyN"][0], "SyN")
        else:
            print(f"  Unexpected warp shape: {warp_data.shape}")
    else:
        print("  No warp field found in SyN result")

    print(f"  Registration time: {reg_time:.2f}s")

    # Clean up temp files
    import os
    for p in result.get("fwdtransforms", []) + result.get("invtransforms", []):
        if os.path.exists(p):
            os.remove(p)
else:
    print("ANTsPy not available — skipping SyN")

## Method 3: VoxelMorph (deep learning)

In [ ]:
if HAS_VXM:
    import os
    os.environ['NEURITE_BACKEND'] = 'pytorch'
    os.environ['VXM_BACKEND'] = 'pytorch'

    import torch
    import voxelmorph as vxm
    import neurite as ne

    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Quick-train a small model on the test pair (+ augmentations)
    model = vxm.nn.models.VxmPairwise(
        ndim=2,
        source_channels=1,
        target_channels=1,
        nb_features=[16, 32, 32, 16],
        integration_steps=0,  # direct regression → more folding
        device=DEVICE,
    ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    img_loss_fn = ne.nn.modules.MSE()
    grad_loss_fn = ne.nn.modules.SpatialGradient('l2')

    # Augment: small random affine jitter
    rng_vxm = np.random.default_rng(42)
    fixed_t = torch.from_numpy(fixed_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)
    moving_t = torch.from_numpy(moving_np).float().unsqueeze(0).unsqueeze(0).to(DEVICE)

    print(f"Training VoxelMorph on {DEVICE}...")
    model.train()
    for epoch in range(150):
        optimizer.zero_grad()
        disp, warped = model(
            moving_t, fixed_t,
            return_warped_source=True,
            return_field_type='displacement',
        )
        loss = img_loss_fn(fixed_t, warped) + 0.01 * grad_loss_fn(disp)
        loss.backward()
        optimizer.step()
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1:>4d}  loss={loss.item():.6f}")

    # Inference
    model.eval()
    with torch.no_grad():
        t0 = time.perf_counter()
        disp, _ = model(
            moving_t, fixed_t,
            return_warped_source=True,
            return_field_type='displacement',
        )
        reg_time = time.perf_counter() - t0

    disp_np = disp.cpu().numpy()[0]  # (2, H, W) → [dy, dx]
    dy = disp_np[0].astype(np.float64)
    dx = disp_np[1].astype(np.float64)
    deformations["VoxelMorph"] = (to_dvfopt(dy, dx), reg_time)
    summarize_dvf(deformations["VoxelMorph"][0], "VoxelMorph")
    print(f"  Inference time: {reg_time:.4f}s")
else:
    print("VoxelMorph not available — skipping")

---
## Correct folding for all methods

In [ ]:
correction_results = {}

for method, (deformation, reg_time) in deformations.items():
    phi_init = np.stack([deformation[1, 0], deformation[2, 0]])
    jac_init = jacobian_det2D(phi_init)
    n_neg_init = int((jac_init <= 0).sum())

    print(f"\n{'='*70}")
    print(f"  {method}  |  neg Jdet: {n_neg_init}  |  reg time: {reg_time:.2f}s")
    print(f"{'='*70}")

    if n_neg_init == 0:
        print("  No folding — skipping correction")
        correction_results[method] = {
            "reg_time": reg_time,
            "n_neg_init": 0,
            "n_neg_final": 0,
            "correction_time": 0.0,
            "l2_err": 0.0,
            "min_jdet_init": float(jac_init.min()),
            "min_jdet_final": float(jac_init.min()),
        }
        continue

    t0 = time.perf_counter()
    phi = iterative_parallel(
        deformation.copy(), verbose=1, threshold=JDET_THRESHOLD
    )
    correction_time = time.perf_counter() - t0

    jac_final = jacobian_det2D(phi)
    n_neg_final = int((jac_final <= 0).sum())
    l2 = float(np.sqrt(np.sum((phi - phi_init) ** 2)))

    correction_results[method] = {
        "reg_time": reg_time,
        "n_neg_init": n_neg_init,
        "n_neg_final": n_neg_final,
        "correction_time": correction_time,
        "l2_err": l2,
        "min_jdet_init": float(jac_init.min()),
        "min_jdet_final": float(jac_final.min()),
        "phi": phi,
    }

    print(f"  Correction: {correction_time:.2f}s  |  neg: {n_neg_init} -> {n_neg_final}  "
          f"|  L2: {l2:.4f}")

    plot_grid_before_after(deformation, phi, title=method)
    plot_neg_jdet_neighborhoods(deformation, phi, title=method)

---
## Summary Table

In [ ]:
print(f"{'Method':<15s}  {'Reg (s)':>8s}  {'Neg init':>10s}  {'Neg final':>10s}  "
      f"{'Corr (s)':>8s}  {'L2 Error':>10s}  {'Min Jdet':>10s}")
print("-" * 82)

for method, r in correction_results.items():
    print(f"{method:<15s}  {r['reg_time']:>8.2f}  {r['n_neg_init']:>10d}  "
          f"{r['n_neg_final']:>10d}  {r['correction_time']:>8.2f}  "
          f"{r['l2_err']:>10.4f}  {r['min_jdet_final']:>+10.6f}")

In [ ]:
# --- Bar chart comparison ---
methods = list(correction_results.keys())
if len(methods) > 0:
    x = np.arange(len(methods))

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    # Neg Jdet: before vs after
    ax = axes[0]
    neg_init = [correction_results[m]["n_neg_init"] for m in methods]
    neg_final = [correction_results[m]["n_neg_final"] for m in methods]
    ax.bar(x - 0.15, neg_init, 0.3, label="Before", color="tab:red", alpha=0.7)
    ax.bar(x + 0.15, neg_final, 0.3, label="After", color="tab:green", alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("Neg Jdet pixels")
    ax.set_title("Folding: Before vs After Correction")
    ax.legend()

    # Registration time vs correction time
    ax = axes[1]
    reg_t = [correction_results[m]["reg_time"] for m in methods]
    corr_t = [correction_results[m]["correction_time"] for m in methods]
    ax.bar(x - 0.15, reg_t, 0.3, label="Registration", color="tab:blue", alpha=0.7)
    ax.bar(x + 0.15, corr_t, 0.3, label="Correction", color="tab:orange", alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(methods, rotation=20, ha="right")
    ax.set_ylabel("Time (s)")
    ax.set_title("Registration vs Correction Time")
    ax.legend()

    # L2 deviation
    ax = axes[2]
    l2s = [correction_results[m]["l2_err"] for m in methods]
    ax.bar(methods, l2s, color="tab:purple", alpha=0.7)
    ax.set_ylabel("L2 Error")
    ax.set_title("Correction L2 Deviation")
    plt.xticks(rotation=20, ha="right")

    plt.suptitle("Registration Method Comparison", fontsize=13)
    plt.tight_layout()
    plt.show()

## Summary

Key takeaways:
- **Diffeomorphic methods** (SyN, diffeomorphic Demons) tend to produce fewer or no folds
- **Direct methods** (B-spline, VoxelMorph with `integration_steps=0`) are more prone to folding
- **Correction overhead** is typically small relative to registration time
- **L2 deviation** shows how much the displacement field had to change — lower is better